# Hybrid Beam Forming using a Multilayer Perceptron 

## Load Helpers 

In [2]:
import numpy as np 
import matplotlib.pyplot as plt 
import json 
import time 
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader, random_split 

import mitsuba as mi
# mi.set_variant("cuda_ad_mono_polarized")
# mi.set_variant("llvm_ad_rgb"); 
# print(mi.variant()); 

from sionna.rt import PathSolver

import os 
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "scripts")); 
sys.path.append(str(Path.cwd().parent / "outputs")); 
from dataset_handler import DatasetHandler 
from mlp import MLP 
from generate_raynet_dataset_v3 import (
    CONFIG,
    setup_scene,
    make_tx_positions,
    make_candidate_configs,
    sample_two_users_for_target_class,
    trace_four_links,
    score_all_joint_configs,
    target_class_passes,
    thermal_noise_watts,
    dbm_to_watts,
) 

### Parameters 

In [3]:
MODEL_CONFIG = {
    "input": 8, 
    "h1_dim": 128, 
    "h2_dim": 128, 
    "h3_dim": 64, 
    "out_1": len(CONFIG["sector_angles_deg"]),
    "out_2": CONFIG["num_codebooks"]
}; 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); 
print(f"Device: {DEVICE}"); 

BATCH_SIZE = 32; 
EPOCHS = 100; 
LR = 1e-2; 
lr_str = str(LR).replace("0.", ""); 

Device: cpu


## Train MLP 

In [ ]:
def plot_training_curves(train_losses, val_metrics, model_name): # Helper to plot train loss and validation error 

    epochs = range(1, len(train_losses) + 1); 

    plt.figure(figsize=(12, 6)); 

    # Train loss
    plt.subplot(1, 2, 1); 
    plt.plot(epochs, train_losses, label = "Train Loss"); 
    plt.xlabel("Epoch"); 
    plt.ylabel("Loss"); 
    plt.title("Training Loss"); 
    plt.legend(); 

    # Validation error 
    plt.subplot(1, 2, 2); 
    plt.plot(epochs, val_metrics["tx0_sector"], label = "TX0 Sector Error"); 
    plt.plot(epochs, val_metrics["tx1_sector"], label = "TX1 Sector Error"); 
    plt.plot(epochs, val_metrics["tx0_codebook"], label = "TX0 Codebook Error"); 
    plt.plot(epochs, val_metrics["tx1_codebook"], label = "TX1 Codebook Error"); 

    plt.xlabel("Epoch"); 
    plt.ylabel("Error Rate"); 
    plt.title("Validation Errors"); 
    plt.legend(); 

    plt.tight_layout(); 
    plt_path = f"../figures/{model_name}_loss_curves.png"; 
    plt.savefig(plt_path); 
    plt.show(); 
    plt.close(); 


def train(trainset, model_name): 

    dataset = DatasetHandler(trainset); 

    train_size = int(0.8 * len(dataset)); 
    val_size = len(dataset) - train_size; 

    train_set, val_set = random_split(
        dataset,
        [train_size, val_size]
    ); 

    train_loader = DataLoader(
        train_set,
        batch_size = BATCH_SIZE,
        shuffle = True
    ); 

    val_loader = DataLoader(
        val_set,
        batch_size = BATCH_SIZE,
        shuffle = False
    ); 

    model = MLP(
        input = MODEL_CONFIG["input"],
        h1_dim = MODEL_CONFIG["h1_dim"],
        h2_dim = MODEL_CONFIG["h2_dim"],
        h3_dim = MODEL_CONFIG["h3_dim"],
        out_1 = MODEL_CONFIG["out_1"],
        out_2 = MODEL_CONFIG["out_2"],
    ).to(DEVICE); 

    criterion = nn.CrossEntropyLoss(); 

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr = LR
    ); 

    train_losses = []; 

    val_tx0_sector = []; 
    val_tx1_sector = []; 
    val_tx0_codebook = []; 
    val_tx1_codebook = []; 

    for epoch in range(EPOCHS):

        model.train(); 

        running_loss = 0.0; 

        for (
            x,
            tx0_sector,
            tx1_sector,
            tx0_codebook,
            tx1_codebook
        ) in train_loader:

            x = x.to(DEVICE); 

            tx0_sector = tx0_sector.to(DEVICE); 
            tx1_sector = tx1_sector.to(DEVICE); 

            tx0_codebook = tx0_codebook.to(DEVICE); 
            tx1_codebook = tx1_codebook.to(DEVICE); 

            optimizer.zero_grad(); 

            s0, s1, c0, c1 = model(x); 

            loss_s0 = criterion(s0, tx0_sector); 
            loss_s1 = criterion(s1, tx1_sector); 

            loss_c0 = criterion(c0, tx0_codebook); 
            loss_c1 = criterion(c1, tx1_codebook); 

            loss = (
                loss_s0
                + loss_s1
                + loss_c0
                + loss_c1
            ); 

            loss.backward(); 
            optimizer.step(); 

            running_loss += loss.item(); 

        avg_train_loss = running_loss / len(train_loader); 
        train_losses.append(avg_train_loss); 

        print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Loss: {avg_train_loss:.4f}"
        ); 

        # Test on validation set 
        model.eval(); 

        correct_tx0_s = 0; 
        total_tx0_s = 0; 

        correct_tx1_s = 0; 
        total_tx1_s = 0; 

        correct_tx0_c = 0; 
        total_tx0_c = 0; 

        correct_tx1_c = 0; 
        total_tx1_c = 0; 

        with torch.no_grad(): 
            for (
                x,
                tx0_sector,
                tx1_sector,
                tx0_codebook,
                tx1_codebook
            ) in val_loader: 

                x = x.to(DEVICE); 
                tx0_sector = tx0_sector.to(DEVICE); 
                tx1_sector = tx1_sector.to(DEVICE); 
                tx0_codebook = tx0_codebook.to(DEVICE); 
                tx1_codebook = tx1_codebook.to(DEVICE); 

                s0, s1, c0, c1 = model(x); 

                pred_s0 = torch.argmax(s0, dim = 1); 
                pred_s1 = torch.argmax(s1, dim = 1); 
                pred_c0 = torch.argmax(c0, dim = 1); 
                pred_c1 = torch.argmax(c1, dim = 1); 

                correct_tx0_s += (pred_s0 == tx0_sector).sum().item(); 
                correct_tx1_s += (pred_s1 == tx1_sector).sum().item(); 
                correct_tx0_c += (pred_c0 == tx0_codebook).sum().item(); 
                correct_tx1_c += (pred_c1 == tx1_codebook).sum().item(); 

                total_tx0_s += tx0_sector.size(0); 
                total_tx1_s += tx1_sector.size(0); 
                total_tx0_c += tx0_codebook.size(0); 
                total_tx1_c += tx1_codebook.size(0); 

        # Compute validation error rates
        val_tx0_sector.append(1 - correct_tx0_s / total_tx0_s); 
        val_tx1_sector.append(1 - correct_tx1_s / total_tx1_s); 
        val_tx0_codebook.append(1 - correct_tx0_c / total_tx0_c); 
        val_tx1_codebook.append(1 - correct_tx1_c / total_tx1_c); 


    torch.save(
        model.state_dict(),
        f"../models/{model_name}.pth" 
    ); 

    model_info = {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
    
        "architecture": {
            "input_dim": MODEL_CONFIG["input"],
            "h1_dim": MODEL_CONFIG["h1_dim"],
            "h2_dim": MODEL_CONFIG["h2_dim"],
            "h3_dim": MODEL_CONFIG["h3_dim"],
            "out_1": MODEL_CONFIG["out_1"],
            "out_2": MODEL_CONFIG["out_2"],
        } 
    }; 

    with open(f"../models/{model_name}.json", "w") as f:
        json.dump(model_info, f, indent = 4); 

    print("Model saved."); 

    val_metrics = {
        "tx0_sector": val_tx0_sector,
        "tx1_sector": val_tx1_sector,
        "tx0_codebook": val_tx0_codebook,
        "tx1_codebook": val_tx1_codebook,
    }; 

    plot_training_curves(train_losses, val_metrics, model_name); 

bs_list = [32, 64, 128]; 
lr_list = [1e-2, 1e-3, 1e-4]; 

for i in range(len(lr_list)): 
    for j in range(len(bs_list)): 
        LR = lr_list[i]; 
        BATCH_SIZE = bs_list[j]; 
        train("../Datasets/dataset_8sector_3book_full_100.csv", f"mlp_{i * 3 + j}"); 


## Test MLP

In [ ]:
def test(testset): 

    dataset = DatasetHandler(testset); 

    loader = DataLoader(
        dataset,
        batch_size = BATCH_SIZE, 
        shuffle = False
    ); 

    model = MLP(
        input = MODEL_CONFIG["input"],
        h1_dim = MODEL_CONFIG["h1_dim"],
        h2_dim = MODEL_CONFIG["h2_dim"],
        h3_dim = MODEL_CONFIG["h3_dim"],
        out_1 = MODEL_CONFIG["out_1"],
        out_2 = MODEL_CONFIG["out_2"],
    ).to(DEVICE); 

    model.load_state_dict(
        torch.load(
            f"../models/mlp_4.pth", 
            map_location = DEVICE
        )
    ); 

    model.to(DEVICE); 
    model.eval(); 

    correct_s0 = 0; 
    correct_s1 = 0; 

    correct_c0 = 0; 
    correct_c1 = 0; 

    total = 0; 

    with torch.no_grad():

        for (
            x,
            tx0_sector,
            tx1_sector,
            tx0_codebook,
            tx1_codebook
        ) in loader:

            x = x.to(DEVICE); 

            s0, s1, c0, c1 = model(x); 

            pred_s0 = torch.argmax(s0, dim = 1); 
            pred_s1 = torch.argmax(s1, dim = 1); 

            pred_c0 = torch.argmax(c0, dim = 1); 
            pred_c1 = torch.argmax(c1, dim = 1); 

            correct_s0 += (
                pred_s0.cpu() == tx0_sector
            ).sum().item(); 

            correct_s1 += (
                pred_s1.cpu() == tx1_sector
            ).sum().item(); 

            correct_c0 += (
                pred_c0.cpu() == tx0_codebook
            ).sum().item(); 

            correct_c1 += (
                pred_c1.cpu() == tx1_codebook
            ).sum().item(); 

            total += x.size(0); 

    print(
        f"TX0 Sector Accuracy: "
        f"{100 * correct_s0 / total:.2f}%"
    ); 

    print(
        f"TX1 Sector Accuracy: "
        f"{100 * correct_s1 / total:.2f}%"
    ); 

    print(
        f"TX0 Codebook Accuracy: "
        f"{100 * correct_c0 / total:.2f}%"
    ); 

    print(
        f"TX1 Codebook Accuracy: "
        f"{100 * correct_c1 / total:.2f}%"
    ); 

test("../Datasets/dataset_test_3book_full_10.csv"); 

## Test MLP on Random User Pairs 
Generate two users in an environment and check if our MLP can determine the proper configuration to give the users the required throughput and maintain a sufficient signal to noise ratio. 

In [124]:
def test_single_pair(model, scene, solver, candidates, verbose = False): 
    tx0_pos, tx1_pos = make_tx_positions(); 
    rng = np.random.default_rng(); 

    # Generate two random users 
    tx0_target = rng.choice(CONFIG["sector_angles_deg"]); 
    tx1_target = rng.choice(CONFIG["sector_angles_deg"]); 

    users = sample_two_users_for_target_class(
        rng,
        tx0_target_angle=tx0_target,
        tx1_target_angle=tx1_target,
    ); 

    if users is None:
        return None; 

    users["u1_required_rate_bpshz"] = rng.uniform(1, 5); 
    users["u2_required_rate_bpshz"] = rng.uniform(1, 5); 

    links = trace_four_links(
        scene = scene,
        p_solver = solver,
        tx0_pos = tx0_pos,
        tx1_pos = tx1_pos,
        users = users,
        sample_seed = 12345,
    ); 


    # Build feature vector 
    noise = thermal_noise_watts(); 

    p_tx0 = dbm_to_watts(CONFIG["tx0_power_dbm"]); 
    p_tx1 = dbm_to_watts(CONFIG["tx1_power_dbm"]); 

    pilot_snr_u1 = max(
        p_tx0 * links["h11"]["power_linear"],
        p_tx1 * links["h21"]["power_linear"],
    ) / noise; 

    pilot_snr_u2 = max(
        p_tx0 * links["h12"]["power_linear"],
        p_tx1 * links["h22"]["power_linear"],
    ) / noise; 

    u1_pilot_snr_db = 10 * np.log10(pilot_snr_u1 + 1e-30); 
    u2_pilot_snr_db = 10 * np.log10(pilot_snr_u2 + 1e-30); 

    x = np.array([
        users["u1_distance_m"],
        users["u1_angle_deg"],
        users["u1_required_rate_bpshz"],
        u1_pilot_snr_db,

        users["u2_distance_m"],
        users["u2_angle_deg"],
        users["u2_required_rate_bpshz"],
        u2_pilot_snr_db,
    ], dtype=np.float32); 

    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(DEVICE); 


    # Use our MLP to determine the best code sector and beam indices ! 
    mlp_prev_t = time.perf_counter(); 
    with torch.no_grad():
        logits_s0, logits_s1, logits_c0, logits_c1 = model(x); 

        tx0_sector = logits_s0.argmax(dim=1).item(); 
        tx1_sector = logits_s1.argmax(dim=1).item(); 

        tx0_codebook = logits_c0.argmax(dim=1).item(); 
        tx1_codebook = logits_c1.argmax(dim=1).item(); 
    mlp_final_t = time.perf_counter(); 
    mlp_t = mlp_final_t - mlp_prev_t; 

    if verbose: 
        print(); 
        print("Predicted Configuration"); 
        print("----------------------"); 
        print(f"TX0 sector   = {tx0_sector}"); 
        print(f"TX0 codebook = {tx0_codebook}"); 
        print(f"TX1 sector   = {tx1_sector}"); 
        print(f"TX1 codebook = {tx1_codebook}"); 
        print(f"Time: {mlp_t:.10e}"); 

    pred_tx0_idx = candidates[(candidates["sector_idx"] == tx0_sector) & (candidates["codebook_idx"] == tx0_codebook)].index[0]; 
    pred_tx1_idx = candidates[(candidates["sector_idx"] == tx1_sector) & (candidates["codebook_idx"] == tx1_codebook)].index[0]; 


    # Evaluate predicted configuration 
    search_prev_t = time.perf_counter(); 
    score_data = score_all_joint_configs(
        users = users,
        links = links,
        candidates = candidates,
    ); 
    search_final_t = time.perf_counter(); 
    search_t = search_final_t - search_prev_t; 

    passes, metrics = target_class_passes(
        score_data = score_data, 
        users = users,
        target_tx0_idx = pred_tx0_idx,
        target_tx1_idx = pred_tx1_idx,
    ); 

    # Check optimality 
    best_tx0_idx = score_data["best_tx0_idx"]; 
    best_tx1_idx = score_data["best_tx1_idx"]; 

    best_tx0 = candidates.iloc[best_tx0_idx]; 
    best_tx1 = candidates.iloc[best_tx1_idx]; 

    if verbose: 
        print(); 
        print("Optimal Configuration"); 
        print("----------------------"); 
        print(
            f"TX0: sector = {best_tx0['sector_idx']} "
            f"codebook = {best_tx0['codebook_idx']}"
        ); 

        print(
            f"TX1: sector = {best_tx1['sector_idx']} "
            f"codebook = {best_tx1['codebook_idx']}"
        ); 
        print(f"time: {search_t:.10e}"); 

        print(); 
        print("Metrics"); 
        print("----------------------"); 

        for k, v in metrics.items():
            print(f"{k}: {v}"); 

        print(); 
        print("Prediction passes thresholds:", passes); 

    return passes, mlp_t, search_t; 


# Load model 
model = MLP(
    input = MODEL_CONFIG["input"],
    h1_dim = MODEL_CONFIG["h1_dim"],
    h2_dim = MODEL_CONFIG["h2_dim"],
    h3_dim = MODEL_CONFIG["h3_dim"],
    out_1 = MODEL_CONFIG["out_1"],
    out_2 = MODEL_CONFIG["out_2"],
); 

model.load_state_dict(torch.load("../models/mlp_4.pth", map_location=DEVICE)); 

model.eval(); 
model.to(DEVICE); 

# Setup environment 
scene = setup_scene(); 
solver = PathSolver(); 
candidates = make_candidate_configs(); 

test_single_pair(model, scene, solver, candidates, verbose = True); 


Predicted Configuration
----------------------
TX0 sector   = 5
TX0 codebook = 1
TX1 sector   = 6
TX1 codebook = 1
Time: 4.0570006240e-04

Optimal Configuration
----------------------
TX0: sector = 5.0 codebook = 1.0
TX1: sector = 6.0 codebook = 1.0
time: 1.7210002989e-04

Metrics
----------------------
target_score: -35.50037002100725
best_score: -35.50037002100725
target_is_best: 1
target_sinr_u1_db: -1.5848728548646855
target_sinr_u2_db: -2.4107833220043755
target_rate_u1_bpshz: 0.7606424510703963
target_rate_u2_bpshz: 0.6544473831193969
target_sum_rate_bpshz: 1.4150898341897933

Prediction passes thresholds: True


In [125]:
# Test our MLP on N user pairs ! 
N = 100; 

success = 0; 
gen_err = 0; 
mlp_total = 0.0; 
search_total = 0.0; 
for _ in range(N): 
    result = test_single_pair(model, scene, solver, candidates); 
 
    if result is None: # Failed to generate a sample, does not count towards incorrect classifications 
        gen_err += 1; 

    else: 
        passed, mlp_t, search_t = result; 
        if passed: 
            success += 1; 
            mlp_total += mlp_t; 
            search_total += search_t; 

print(f"Pass Rate: { success / (N - gen_err) * 100 }%"); 
print(); 
print(f"Average MLP Time: { mlp_total / (N - gen_err)}"); 
print(f"Average Search Time: { search_total / (N - gen_err)}"); 

Pass Rate: 74.74747474747475%

Average MLP Time: 0.00031752626392802207
Average Search Time: 0.0001570949474387247
